# Explainable Malware Detection — Full Training (Colab)

This notebook clones the project's GitHub repo and runs the real training pipeline
on the full EMBER dataset. It does not contain separate logic — it calls the same
`src/` code that lives in the repo, so anything fixed/changed there is picked up
automatically next time this notebook is re-run.

**Runtime:** Runtime -> Change runtime type -> keep CPU (LightGBM doesn't need GPU;
the EMBER download + vectorization step is the slow part, not training).

**Disk:** EMBER's raw archive is ~10GB compressed, ~intel more once extracted +
vectorized. Colab's default disk (100+GB) is enough; do not attempt this in a
small local environment.

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/Rishi-V1/Bubo-AI-.git repo
%cd repo
!pip install -q -r requirements.txt

## 2. Download EMBER-2018

This is the dataset both Base Paper 1 (Al Siam et al.) and Paper 2 (Mohseni et al.)
use, and the one we train on before testing on BODMAS. Official source: Elastic's
EMBER repo. The `ember` pip package includes a helper to download and vectorize it.

In [ ]:
import os

DATA_DIR = "/content/ember2018"
os.makedirs(DATA_DIR, exist_ok=True)

# Official EMBER 2018 dataset (v2 features, 2381-dim), ~800MB compressed.
# Source: https://github.com/elastic/ember
!wget -q --show-progress -O /content/ember_dataset_2018_2.tar.bz2 \
  https://ember.elastic.co/ember_dataset_2018_2.tar.bz2
!tar -xjf /content/ember_dataset_2018_2.tar.bz2 -C /content/
!mv /content/ember2018 {DATA_DIR} 2>/dev/null || true
!ls {DATA_DIR}

In [ ]:
# Vectorize raw JSONL features into the .dat files our load_dataset.py expects.
# This step is the slow one (~20-40 min depending on Colab's assigned machine).
import ember
ember.create_vectorized_features(DATA_DIR, feature_version=2)

## 3. Load the vectorized data using the repo's own loader

In [ ]:
import sys
sys.path.insert(0, "/content/repo")

from src.features.load_dataset import load_ember_subset

# n_samples=None loads everything; set a smaller number first to sanity-check
# the pipeline runs end-to-end before committing to a full multi-hour run.
X_train, y_train = load_ember_subset(DATA_DIR, n_samples=None, split="train")
X_test, y_test = load_ember_subset(DATA_DIR, n_samples=None, split="test")

print(f"Train: {X_train.shape}, malware ratio {y_train.mean():.3f}")
print(f"Test:  {X_test.shape}, malware ratio {y_test.mean():.3f}")

## 4. Train the clustering-aided LightGBM model

In [ ]:
from src.models.clustered_lightgbm import ClusteredLightGBM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time

model = ClusteredLightGBM(n_clusters=3)

t0 = time.time()
model.fit(X_train.values, y_train.values)
print(f"Training took {time.time()-t0:.1f}s")

y_pred = model.predict(X_test.values)
y_proba = model.predict_proba(X_test.values)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1:        {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print()
print("--- Base Paper 1 (Al Siam et al.) reported ---")
print("Accuracy: 0.9116, Precision: 0.8852, Recall: 0.9422, F1: 0.9128, ROC-AUC: 0.974")

## 5. Compare against a flat (non-clustered) LightGBM baseline

This isolates what clustering actually buys us — the number to report is the
delta between this and step 4, not step 4 in isolation.

In [ ]:
import lightgbm as lgb

flat_model = lgb.LGBMClassifier(
    objective="binary", n_estimators=300, num_leaves=64,
    learning_rate=0.05, verbosity=-1, random_state=42,
)
flat_model.fit(X_train.values, y_train.values)
flat_pred = flat_model.predict(X_test.values)
flat_proba = flat_model.predict_proba(X_test.values)[:, 1]

print("--- Flat (non-clustered) LightGBM baseline ---")
print(f"Accuracy:  {accuracy_score(y_test, flat_pred):.4f}")
print(f"F1:        {f1_score(y_test, flat_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, flat_proba):.4f}")

## 6. Save the trained model

Save to Google Drive so it survives the Colab session ending, and so it can be
downloaded and used by the backend later without retraining.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import joblib
import os

SAVE_DIR = "/content/drive/MyDrive/explainable-malware-detection/models"
os.makedirs(SAVE_DIR, exist_ok=True)
joblib.dump(model, os.path.join(SAVE_DIR, "clustered_lightgbm_ember.joblib"))
print(f"Saved to {SAVE_DIR}")

## Next: BODMAS cross-dataset evaluation

Once this model is trained and saved, the next notebook step (to be added once
`src/evaluation/cross_dataset.py` exists in the repo) loads BODMAS separately
and tests this *same frozen model* on it with no retraining — that's the actual
novelty experiment. Re-clone/pull the repo before running that step to pick up
the new evaluation code.